# Kunz 2025 · inner speech T12, T15, T16

## Prepare the data

Run the commands below from the repository root with `bci-repro` active. See the [reader guide](../../docs/20261124_repository_migration/reader_workflow.md) for environment and English pronunciation resource setup.

### 1. Download

Download `sentenceData.zip` from [Dryad](https://doi.org/10.5061/dryad.gf1vhhn1j). Create `studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/` and save the download as `studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/dryad_gf1vhhn1j__sentenceData.zip`.

### 2. Extract

Extract the archive into `studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j/`.

```text
python -m zipfile -e "studies/20250821_kunz/data/raw/dryad_gf1vhhn1j/dryad_gf1vhhn1j__sentenceData.zip" "studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j"
```

After extraction, check for `studies/20250821_kunz/data/extracted/dryad_gf1vhhn1j/sentenceData/`. It contains `sentences_readme.txt` and the per-session `.mat` files.

### 3. Prepare the model input

Convert the released sentence `.mat` files to the HDF5 layout used by the trainer.

```text
python shared/tools/setup_upstream.py --study 20250821_kunz
python studies/20250821_kunz/code/prepare_data.py
```

**Output** `studies/20250821_kunz/data/derived/dryad_gf1vhhn1j/hdf5/trainer_input/`. The converter writes `data_train.hdf5` for open-loop inputs and `data_val.hdf5` for closed-loop evaluation inputs. The session index is `studies/20250821_kunz/data/derived/dryad_gf1vhhn1j/hdf5/dryad_gf1vhhn1j__session_description.csv`.

### 4. Export the common session file

Write one session, condition, and vocabulary file to the session format shared by every study in this repository. The file stores each sentence with the features as released, at the bin size the file records, and records the `.mat` file, the go-period rows, and the block the sentence came from. This example exports the closed-loop imagined-speech evaluation of T12 on November 28, 2023.

```text
python studies/20250821_kunz/code/export_sessions.py --session t12.2023.11.28_imagined_50words_eval
```

**Output** `studies/20250821_kunz/data/processed/t12/dryad_gf1vhhn1j_2023-11-28_imagined_50words_eval.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its publisher file with one command.

```text
python shared/tools/validate_session.py studies/20250821_kunz/data/processed/t12/dryad_gf1vhhn1j_2023-11-28_imagined_50words_eval.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 100/100 trials re-read and identical`.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20250821_kunz/data/processed/t12/dryad_gf1vhhn1j_2023-11-28_imagined_50words_eval.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = session.trials[0]
        print(trial.trial_id, "·", trial.target("sentence"))
        for name in trial.names("neural"):
            array = trial.neural(name)
            print(f"  neural/{name}: {array.shape} {array.dtype}, "
                  f"{trial.group[f'neural/{name}'].attrs['bin_ms']} ms bins")
        labels = session.label_sets["phonemes"]
        print("  phonemes:", " ".join(labels[i].strip() for i in trial.target("phonemes")[:10]))
        print(f"  {trial.attrs['condition']} speech, {trial.attrs['loop']} loop, block {trial.attrs['source_block']}")
else:
    print("Session file not found. Run the export command above to create it.")

t12.2023.11.28_imagined_50words_eval: 100 trials
t12.2023.11.28_imagined_50words_eval/data/0000 · I need success.
  neural/spike_pow: (207, 128) float64, 20 ms bins
  neural/tx: (207, 128) uint8, 20 ms bins
  phonemes: AY | N IY D | S AH K S
  imagined speech, closed loop, block 11


## Read the archived results

These are the archived mixed-training evaluations. The three arms remain available for supplementary analysis. Earlier Figure 2f OPT weights were selected on the full archived evaluation set.

This walkthrough uses the checked-in archive and takes no data download.

**Which trials are scored, and how to compare with the paper.** The paper's evaluation split is the one the release ships; this pipeline reuses it, so a fresh `trials.csv` covers the same trials as the archived score. The split columns are documented in [`data/splits/README.md`](data/splits/README.md), the paper-figure selection in the [evaluation protocol](docs/20261124_evaluation_protocol/README.md), and the fresh-vs-archived WER comparison recipe in the [fresh-machine guide](../../fresh_machine_setup.md) — match the score-bearing run length before comparing WERs with the paper.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
arm = "Kunz 2025 (inner, T12)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

for participant in ("T15", "T16"):
    name = f"Kunz 2025 (inner, {participant})"
    inputs = benchmark["figure3_per_sources"][name]
    summary = build(name, root / inputs["greedy"], root / inputs["decode"])
    print(participant, summary["pooled"])

t12.2023.11.28_imagined_50words_eval · I need success.
Word hypothesis: YOU KNOW WHAT
Trial word edits: 3/3
Kunz 2025 (inner, T12):  1 sessions | pooled PER 61.40% | WER 122.11%
t12.2023.11.28_imagined_50words_eval: 100 trials, PER 61.40%, WER 122.11%
1 sessions, 100 evaluated trials; pooled {'per': 61.4, 'wer': 122.11}
Kunz 2025 (inner, T15):  3 sessions | pooled PER 59.67% | WER 80.86%
T15 {'per': 59.67, 'wer': 80.86}
Kunz 2025 (inner, T16):  2 sessions | pooled PER 61.49% | WER 95.53%
T16 {'per': 61.49, 'wer': 95.53}


## Reproduce and score

[Common reader guide](../../docs/20261124_repository_migration/reader_workflow.md) · [Study evaluation protocol](docs/20261124_evaluation_protocol/README.md)

### Train and infer

Run from PowerShell at the repository root after the trainer-input conversion. Start with the T12 mixed arm. For T15, T16, imagined, or attempted, change the participant and arm in both the run label and the commands.

```powershell
$study = "studies/20250821_kunz"
$date = Get-Date -Format yyyyMMdd
$run = "$study/results/${date}_dryad_gf1vhhn1j__hdf5__ctc_greedy__t12_mixed_reader/runtime"
python "$study/code/train.py" t12 mixed 1000 "$run" 500
python "$study/code/infer.py" "$run"
```

### Calculate metrics

```powershell
python "$study/code/evaluate.py" "$run" t12
```

Read the pooled greedy PER printed by the script and the counts in `val_greedy_per_trial.csv`. For WER, follow [speech word decoding in WSL](../../docs/20261124_repository_migration/reader_workflow.md#speech-word-error-rate-in-wsl) with the Kunz row and the new `kunz_logits.hdf5`.

For the separate T16 block holdout experiment, `code/make_t16_holdout.py` writes the fold inputs under `data/derived/dryad_gf1vhhn1j/hdf5/t16_holdout/`. See the [code guide](code/README.md) for the experiment settings.
